# Modul 13 · Kausale Inferenz mit Beobachtungsdaten
**Data Science 1** · linkit courses

Was tun, wenn man nicht randomisieren kann? Das Skript wertet die Schulung der Qualitätssicherung und den Workshop zur Rüstzeitverkürzung an Linie 2 aus. Du vergleichst die Gruppen vor der Schulung mit standardisierten Differenzen, findest das Simpson-Paradox in den Nordwerk-Daten und prüfst das Hintertürkriterium mit einer eigenen Funktion. Du rechnest die Adjustierung in Prozentpunkten, siehst Mediator und Collider wirken, stratifizierst und gewichtest nach dem Propensity Score für die Geschulten, ziehst einen Bootstrap und matchst mit drei Partnern. Beim Workshop trennst du nach Art des Wechsels, misst die Bremse durch das neue Auftragssystem und rechnest Placebo-Linien. Am Ende stehen E-Wert und Ergebnisbericht.

**Lernziele**
1. Kausale Fragen mit potenziellen Ergebnissen und Kausalgraphen formulieren
2. Störgrößen erkennen und durch Adjustierung berücksichtigen
3. Erkennen, warum Mediatoren und Collider nicht adjustiert werden dürfen
4. Propensity Score schätzen, Überlappung und Balance prüfen
5. Mit Gewichtung und Matching Effekte schätzen und vergleichen
6. Differenz-in-Differenzen anwenden und parallele Trends prüfen
7. Annahmen benennen, mit Placebo-Tests prüfen und ehrlich berichten

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Der erste Import lädt im Browser SciPy, statsmodels und scikit-learn, das dauert einmalig etwa 15 Sekunden.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Die TODOs bauen nur selten aufeinander auf. Wo ein späteres TODO ein früheres Ergebnis braucht, springt eine Vorgabe-Zelle ein, solange das frühere offen ist. Du kannst die TODOs also auch einzeln bearbeiten.
- Mindestziel sind TODO 1 bis 13. Die beiden Bonusaufgaben vertiefen.
- Aller Zufall hängt an `SEED = 13` in der Zelle nach den Importen. Der Bootstrap in TODO 8 legt seinen Generator mit `np.random.default_rng(SEED)` an, die logistische Regression bekommt `random_state=SEED`. Die Selbsttests lassen so viel Spielraum, dass jeder richtige Weg auch mit einem anderen Seed besteht.
- Gib Ergebnisse mit `float(...)` oder `round(...)` aus, dann steht im Browser und in Colab dasselbe da. Jede Grafikzelle endet mit `plt.show()`.
- `schulung.csv` ist pseudonymisiert. Werte nur Gruppen aus, nie einzelne Personen.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["schulung.csv", "ruestvorgaenge.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Ohne Zufall: das Problem der Störgrößen
*Lerneinheit 1 · Skript Abschnitt 1*

Wer zur Schulung ging, haben die Schichtleitungen entschieden. Im Skript liegen die Geschulten 2026 bei 1,606 % Fehlerquote, die Ungeschulten bei 1,552 %, und schon 2025 hatten die Geschulten mehr Fehler. Eine Größe, die Zuteilung und Ergebnis zugleich beeinflusst, heißt **Störgröße**. Bevor du einen Vergleich deutest, vergleichst du die Gruppen vor der Maßnahme.

Die nächste Zelle lädt die Schulung wie im Skript mit den Fehlerquoten in Prozent, der bei 15 Jahren abgeschnittenen Erfahrung `erfahrung_15` und dem Logarithmus der Fehlerquote 2025 `log_quote_2025`. `T` enthält die Schulung als Array aus 0 und 1. Die Funktion `smd` stammt aus Abschnitt 4.4 des Skripts: Sie berechnet die **standardisierte Differenz** geschult minus ungeschult,

$$d = \frac{\bar x_{\text{geschult}} - \bar x_{\text{ungeschult}}}{\sqrt{(s^2_{\text{geschult}} + s^2_{\text{ungeschult}})/2}},$$

auf Wunsch mit Gewichten im Zähler. Beträge unter 0,1 gelten als ausgeglichen. Die Zelle danach legt den Seed fest.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression

schulung = pd.read_csv("daten/schulung.csv", dtype={"linie": str})
schulung["quote_2025"] = 100 * schulung["fehler_2025"] / schulung["teile_2025"]
schulung["quote_2026"] = 100 * schulung["fehler_2026"] / schulung["teile_2026"]
schulung["erfahrung_15"] = schulung["erfahrung_jahre"].clip(upper=15)
schulung["log_quote_2025"] = np.log(schulung["quote_2025"] + 0.2)
T = schulung["schulung"].to_numpy()


def smd(x, g, w=None):
    """Standardisierte Differenz geschult minus ungeschult, Nenner aus den ungewichteten Gruppen."""
    w = np.ones(len(x)) if w is None else np.asarray(w)
    x, g = np.asarray(x, dtype=float), np.asarray(g)
    nenner = np.sqrt((x[g == 1].var(ddof=1) + x[g == 0].var(ddof=1)) / 2)
    return (np.average(x[g == 1], weights=w[g == 1]) - np.average(x[g == 0], weights=w[g == 0])) / nenner


print(schulung.shape, int(T.sum()), round(schulung["quote_2026"].mean(), 3))

In [ ]:
# Ein Seed für alles: Der Bootstrap legt seinen Generator mit np.random.default_rng(SEED) an, die logistische Regression bekommt random_state=SEED.
SEED = 13

### TODO 1 · Wer wurde geschult?
Das Skript vergleicht Mittelwerte in Jahren und Prozent. Die standardisierte Differenz misst alle Größen in Standardabweichungen und macht sie vergleichbar.
1. `vorher_smd`: eine Series mit dem Index `erfahrung_jahre`, `quote_2025`, `teile_2025`, `nacht`, `linie_2`, `linie_4` (in dieser Reihenfolge) und der standardisierten Differenz `smd(x, T)` der jeweiligen Größe. Für `nacht`, `linie_2` und `linie_4` nimmst du Anzeiger, etwa `schulung["schicht"] == "Nacht"` und `schulung["linie"] == "2"`.

In [ ]:
# TODO 1
vorher_smd = None

print(None if vorher_smd is None else vorher_smd.round(3).to_dict())

In [ ]:
check("TODO 1 standardisierte Differenzen", lambda: list(vorher_smd.index) == ["erfahrung_jahre", "quote_2025", "teile_2025", "nacht", "linie_2", "linie_4"]
      and np.allclose(vorher_smd, [0.3452, 0.4588, 0.0363, -0.0466, 0.6202, -0.4101], atol=5e-4))
check("TODO 1 große und kleine Unterschiede", lambda: bool((vorher_smd[["erfahrung_jahre", "quote_2025", "linie_2", "linie_4"]].abs() > 0.1).all())
      and bool((vorher_smd[["teile_2025", "nacht"]].abs() < 0.1).all()))

Am weitesten auseinander liegen Linie 2 mit 0,620, die Fehlerquote 2025 mit 0,459, Linie 4 mit −0,410 und die Erfahrung mit 0,345 Standardabweichungen, alle weit über 0,1. Die Zahl der geprüften Teile unterscheidet sich mit 0,036 kaum, die Nachtschicht mit −0,047 ebenso. Die Auswahl hängt also mit Linie, Fehlern und Erfahrung zusammen. Im Skript stehen für die abgeschnittene Erfahrung und den Logarithmus der Fehlerquote 0,460 und 0,492: Die Umformung ändert die Größe der Differenz, ihre Richtung bleibt.

### TODO 2 · Simpson-Tabelle nach Fehlerquote 2025
Im Skript trennt die Schicht die Gruppen kaum. Teile die Personen stattdessen nach der Fehlerquote 2025 in zwei Hälften.
1. `simpson`: eine Tabelle mit den Zeilen `unten` und `oben` (Fehlerquote 2025 bis zum Median und darüber) und den Spalten `ungeschult` und `geschult` (mittlere Fehlerquote 2026 in Prozent) sowie `n_ungeschult` und `n_geschult` (Zahl der Personen).
2. `simpson_diff`: der Unterschied geschult minus ungeschult je Hälfte, gemittelt mit der Zahl der Personen je Hälfte als Gewicht, in Prozentpunkten.

In [ ]:
# TODO 2
simpson = None
simpson_diff = None

print(round(schulung["quote_2025"].median(), 3))
print(None if simpson is None else simpson.round(3))
print(None if simpson_diff is None else round(simpson_diff, 3))

In [ ]:
check("TODO 2 Tabelle", lambda: list(simpson.index) == ["unten", "oben"] and list(simpson.columns) == ["ungeschult", "geschult", "n_ungeschult", "n_geschult"]
      and simpson[["n_ungeschult", "n_geschult"]].to_numpy().tolist() == [[50, 12], [40, 22]]
      and np.allclose(simpson[["ungeschult", "geschult"]].to_numpy(), [[1.006, 0.9152], [2.2352, 1.9833]], atol=5e-4))
check("TODO 2 gewichteter Unterschied", lambda: abs(simpson_diff - (-0.1713)) < 5e-4 and bool((simpson["geschult"] < simpson["ungeschult"]).all()))

Der Median der Fehlerquote 2025 liegt bei 1,430 %. In der unteren Hälfte liegen die 12 Geschulten 2026 bei 0,915 %, die 50 Ungeschulten bei 1,006 %. In der oberen Hälfte liegen die 22 Geschulten bei 1,983 %, die 40 Ungeschulten bei 2,235 %. In beiden Hälften sind die Geschulten besser, gewichtet um 0,171 Prozentpunkte. Zusammengenommen liegen sie 0,054 Prozentpunkte schlechter, weil fast zwei Drittel von ihnen aus der oberen Hälfte kommen. Das ist das Simpson-Paradox aus dem Skript, diesmal in den Nordwerk-Daten. Zwei Hälften sind eine grobe Einteilung, innerhalb einer Hälfte können sich die Gruppen 2025 noch unterscheiden. Die Regression in Abschnitt 3 nimmt die Fehlerquote 2025 deshalb als stetige Größe auf.

## 2 · Kausalgraphen zeichnen
*Lerneinheit 2 · Skript Abschnitt 2*

Ein **Kausalgraph** schreibt die Annahmen auf, was wovon abhängt. Ein **Hintertürpfad** ist ein Pfad von der Schulung zu den Fehlern 2026, der mit einem Pfeil in die Schulung beginnt. Eine Menge von Größen reicht zur Adjustierung, wenn keine davon eine Folge der Schulung ist und sie jeden Hintertürpfad blockiert: Auf dem Pfad liegt eine Gabel oder Kette, deren mittlerer Knoten in der Menge ist, oder ein Collider, der weder selbst in der Menge ist noch eine Folge in der Menge hat.

Die nächste Zelle legt die Funktionen `pfade` und `collider` aus Abschnitt 2.3 des Skripts an. `pfade` sucht alle Pfade zwischen zwei Knoten, egal in welche Richtung die Pfeile zeigen, `collider` nennt die Collider auf einem Pfad.

In [ ]:
def pfade(pfeile, start, ziel, weg=None):
    """Alle Pfade von start nach ziel, egal in welche Richtung die Pfeile zeigen."""
    weg = weg or [start]
    if start == ziel:
        return [weg]
    nachbarn = [b for a, b in pfeile if a == start] + [a for a, b in pfeile if b == start]
    return [p for n in nachbarn if n not in weg for p in pfade(pfeile, n, ziel, weg + [n])]


def collider(pfeile, p):
    return [p[i] for i in range(1, len(p) - 1) if (p[i - 1], p[i]) in pfeile and (p[i + 1], p[i]) in pfeile]


beispiel = [("Alter", "Wartung"), ("Alter", "Ausfall"), ("Wartung", "Verschleiß"), ("Verschleiß", "Ausfall"),
            ("Wartung", "Meldung"), ("Ausfall", "Meldung")]
print(len(pfade(beispiel, "Wartung", "Ausfall")), collider(beispiel, ["Wartung", "Meldung", "Ausfall"]))

### TODO 3 · Kausalgraph der Schulung als Pfeilliste
Der Graph aus Abschnitt 2.4 des Skripts in Worten: Erfahrung, Schicht und Linie wirken je auf Fehler 2025, Schulung und Fehler 2026. Das unbeobachtete Können wirkt auf Fehler 2025 und Fehler 2026. Fehler 2025 wirkt auf die Schulung. Die Schulung wirkt auf Audit, Fehler 2026 und Einrichter, das Audit auf Fehler 2026 und Fehler 2026 auf Einrichter.
1. `graph_schulung`: die 17 Pfeile als Liste von Paaren `(von, nach)` mit genau diesen Knotennamen: `"Erfahrung"`, `"Schicht"`, `"Linie"`, `"Können"`, `"Fehler 2025"`, `"Schulung"`, `"Audit"`, `"Fehler 2026"`, `"Einrichter"`.
2. `hintertuer`: alle Pfade aus `pfade(graph_schulung, "Schulung", "Fehler 2026")`, die mit einem Pfeil in die Schulung beginnen.
3. `blockiert(pfeile, pfad, menge)`: `True`, wenn die Menge den Pfad blockiert, sonst `False`. Ein mittlerer Knoten blockiert, wenn er kein Collider ist und in der Menge liegt, oder wenn er ein Collider ist und weder er noch eine seiner Folgen in der Menge liegt. Folgen eines Knotens sind alle Knoten, die man von ihm aus entlang der Pfeile erreicht.
4. `menge`: die Adjustierungsmenge aus dem Skript, `{"Erfahrung", "Schicht", "Linie", "Fehler 2025"}`. `offen`: die Hintertürpfade, die `menge` nicht blockiert. `offen_ohne_2025`: dieselbe Liste für die Menge ohne `"Fehler 2025"`.

In [ ]:
# TODO 3
graph_schulung = None
hintertuer = None


def blockiert(pfeile, pfad, menge):
    return None


menge = None
offen = None
offen_ohne_2025 = None

print(None if graph_schulung is None else len(graph_schulung), None if hintertuer is None else len(hintertuer))
print(offen, offen_ohne_2025)

In [ ]:
check("TODO 3 Graph und Hintertürpfade", lambda: len(set(graph_schulung)) == len(graph_schulung) == 17 and ("Fehler 2025", "Schulung") in graph_schulung
      and ("Können", "Schulung") not in graph_schulung and len(pfade(graph_schulung, "Schulung", "Fehler 2026")) == 19
      and len(hintertuer) == 16 and sum(not collider(graph_schulung, p) for p in hintertuer) == 7)
check("TODO 3 Funktion blockiert", lambda: blockiert(graph_schulung, ["Schulung", "Einrichter", "Fehler 2026"], set()) is True
      and blockiert(graph_schulung, ["Schulung", "Einrichter", "Fehler 2026"], {"Einrichter"}) is False
      and blockiert(graph_schulung, ["Schulung", "Erfahrung", "Fehler 2026"], {"Erfahrung"}) is True
      and blockiert(graph_schulung, ["Schulung", "Erfahrung", "Fehler 2026"], set()) is False
      and blockiert(graph_schulung, ["Schulung", "Erfahrung", "Fehler 2025", "Können", "Fehler 2026"], set()) is True
      and blockiert(graph_schulung, ["Schulung", "Erfahrung", "Fehler 2025", "Können", "Fehler 2026"], {"Audit"}) is False)
check("TODO 3 Adjustierungsmenge", lambda: menge == {"Erfahrung", "Schicht", "Linie", "Fehler 2025"} and offen == []
      and offen_ohne_2025 == [["Schulung", "Fehler 2025", "Können", "Fehler 2026"]])

Von der Schulung zu den Fehlern 2026 führen 19 Pfade, 16 davon sind Hintertürpfade, und 7 dieser 16 haben keinen Collider. Die Menge aus Erfahrung, Schicht, Linie und Fehler 2025 blockiert alle 16. Ohne Fehler 2025 bleibt genau ein Pfad offen: Schulung ← Fehler 2025 ← Können → Fehler 2026, die Hintertür über das unbeobachtete Können. Fehler 2025 ist auf anderen Pfaden selbst ein Collider, etwa Schulung ← Erfahrung → Fehler 2025 ← Können → Fehler 2026. Hält man Fehler 2025 fest, öffnen sich solche Pfade; Erfahrung, Schicht und Linie schließen sie wieder. Mit Fehler 2025 allein blieben deshalb 12 Hintertürpfade offen. Audit und Einrichter sind Folgen der Schulung und gehören nicht in die Menge, auch wenn sie dort keinen Hintertürpfad öffnen würden.

## 3 · Adjustieren und was man nicht adjustieren darf
*Lerneinheit 3 · Skript Abschnitt 3*

Im Skript schätzt eine Poisson-Regression mit dem Logarithmus der geprüften Teile als Offset das Verhältnis der Fehlerraten geschult zu ungeschult, mit robusten Standardfehlern. Mit allen vier Störgrößen ist es 0,851. Die nächste Zelle legt `verhaeltnis`, `STOERGROESSEN` und `stufen` wie im Skript an und zeigt diese Schätzung mit 95-%-Intervall.

In [ ]:
def verhaeltnis(rechts, daten=schulung):
    """Verhältnis der Fehlerraten geschult zu ungeschult mit 95-%-Intervall (Poisson, Offset, robuste Standardfehler)."""
    m = smf.glm("fehler_2026 ~ " + rechts, data=daten, family=sm.families.Poisson(),
                offset=np.log(daten["teile_2026"])).fit(cov_type="HC0")
    unten, oben = np.exp(m.conf_int().loc["schulung"])
    return round(float(np.exp(m.params["schulung"])), 3), round(float(unten), 3), round(float(oben), 3)


STOERGROESSEN = "erfahrung_15 + C(schicht) + C(linie) + log_quote_2025"
stufen = {"naiv": "schulung", "+ Erfahrung": "schulung + erfahrung_15", "+ Schicht, Linie": "schulung + erfahrung_15 + C(schicht) + C(linie)",
          "+ Fehler 2025": "schulung + " + STOERGROESSEN, "nur Fehler 2025": "schulung + log_quote_2025"}
print(verhaeltnis(stufen["+ Fehler 2025"]))

### TODO 4 · Adjustieren in Prozentpunkten
Ein Verhältnis von 0,851 ist anschaulich, aber wie viele Prozentpunkte Fehlerquote sind das? Ein lineares Modell der Fehlerquote 2026 beantwortet das direkt.
1. `stufen_pp`: eine Tabelle mit einer Zeile je Eintrag von `stufen` (gleiche Namen und Reihenfolge, also auch „nur Fehler 2025“) und den Spalten `Differenz`, `unten` und `oben`. Darin stehen der Koeffizient von `schulung` und sein 95-%-Intervall aus `smf.ols("quote_2026 ~ " + rechts, data=schulung).fit(cov_type="HC1")`, in Prozentpunkten.

In [ ]:
# TODO 4
stufen_pp = None

print(None if stufen_pp is None else stufen_pp.round(3))

In [ ]:
check("TODO 4 Differenzen je Stufe", lambda: list(stufen_pp.index) == list(stufen) and list(stufen_pp.columns) == ["Differenz", "unten", "oben"]
      and np.allclose(stufen_pp["Differenz"], [0.054, 0.1935, 0.0011, -0.3121, -0.3159], atol=5e-4))
check("TODO 4 Intervalle", lambda: np.allclose(stufen_pp.loc["naiv", ["unten", "oben"]].astype(float), [-0.2899, 0.3979], atol=5e-4)
      and np.allclose(stufen_pp.loc["+ Fehler 2025", ["unten", "oben"]].astype(float), [-0.484, -0.1402], atol=5e-4))

Naiv liegen die Geschulten 0,054 Prozentpunkte höher, mit Intervall von −0,290 bis 0,398. Mit der Erfahrung steigt der Unterschied auf 0,194, mit Schicht und Linie fällt er auf 0,001. Erst mit der Fehlerquote 2025 dreht er: −0,312 Prozentpunkte mit Intervall von −0,484 bis −0,140. Die Fehlerquote 2025 allein bringt schon −0,316. Ohne Schulung wäre bei den Geschulten eine Quote von etwa 1,606 + 0,312 = 1,918 % zu erwarten; gemessen daran sind −0,312 Prozentpunkte etwa 16 % weniger, nahe an den 15 % aus dem Verhältnis. Beide Modelle sagen dasselbe in verschiedenen Maßen: Das Verhältnis gilt je Teil und gewichtet Personen mit vielen Teilen stärker, das lineare Modell zählt jede Person gleich. Schreib zu jeder Zahl dazu, ob sie in Prozent oder Prozentpunkten gemeint ist.

### TODO 5 · Mediator und Collider in Prozentpunkten
Im Skript schrumpft das Verhältnis mit dem Audit auf 0,916, unter den Einrichtern verschwindet die Wirkung. Rechne beides mit dem linearen Modell aus TODO 4.
1. `mediator_pp`: Schätzung, untere und obere Grenze als Tupel aus drei Zahlen für das lineare Modell mit allen Störgrößen (`"schulung + " + STOERGROESSEN`) und zusätzlich `audit_abweichungen`.
2. `collider_pp`: ein Dictionary mit den Schlüsseln `"naiv"` und `"voll"` und je einem solchen Tupel, gerechnet nur für die 34 späteren Einrichterinnen und Einrichter (`einrichter_2026 == 1`): einmal nur mit `schulung`, einmal mit allen Störgrößen.

In [ ]:
# TODO 5
mediator_pp = None
collider_pp = None

print(None if mediator_pp is None else np.round(mediator_pp, 3))
print(None if collider_pp is None else {k: np.round(v, 3) for k, v in collider_pp.items()})

In [ ]:
check("TODO 5 Mediator", lambda: len(mediator_pp) == 3 and np.allclose(mediator_pp, [-0.1873, -0.3389, -0.0358], atol=5e-4))
check("TODO 5 Collider", lambda: set(collider_pp) == {"naiv", "voll"} and np.allclose(collider_pp["naiv"], [0.3267, -0.0852, 0.7386], atol=5e-4)
      and np.allclose(collider_pp["voll"], [-0.1073, -0.2929, 0.0784], atol=5e-4))

Mit dem Audit schrumpft der Unterschied von −0,312 auf −0,187 Prozentpunkte. Das Modell vergleicht dann Personen mit gleich vielen Abweichungen im Audit und rechnet den Teil der Wirkung heraus, der über weniger Abweichungen läuft. Unter den 34 Einrichtern liegen die Geschulten naiv 0,327 Prozentpunkte höher, mit allen Störgrößen −0,107 mit Intervall von −0,293 bis 0,078. Die Wirkung schrumpft auf ein Drittel und ist nicht mehr nachgewiesen. Beide Größen wurden nach der Schulung gemessen. In Prozentpunkten zeigt sich dasselbe Muster wie mit dem Verhältnis im Skript.

## 4 · Propensity Score: Gewichten und Matchen
*Lerneinheit 4 · Skript Abschnitt 4*

Der **Propensity Score** $e(x) = P(T_i = 1 \mid X_i = x)$ fasst die Störgrößen in einer Zahl zusammen. Die nächste Zelle schätzt ihn wie im Skript mit einer logistischen Regression ohne nennenswerte Regularisierung (großes `C`) und legt die ATE-Gewichte $w_i = T_i/e(X_i) + (1-T_i)/(1-e(X_i))$ in `schulung["w_ate"]` an. Dazu kommen die Arrays `ps`, `f` (Fehler 2026) und `n` (Teile 2026) und die Funktion `gewichtetes_verhaeltnis` aus dem Skript: Sie teilt je Gruppe die gewichtete Summe der Fehler durch die gewichtete Summe der Teile.

In [ ]:
merkmale = pd.get_dummies(schulung[["schicht", "linie"]], drop_first=True, dtype=float)
merkmale[["erfahrung_15", "log_quote_2025"]] = schulung[["erfahrung_15", "log_quote_2025"]]
modell_ps = LogisticRegression(C=1e6, max_iter=10_000, random_state=SEED).fit(merkmale, schulung["schulung"])
schulung["ps"] = modell_ps.predict_proba(merkmale)[:, 1]
schulung["w_ate"] = T / schulung["ps"] + (1 - T) / (1 - schulung["ps"])
ps, f, n = schulung["ps"].to_numpy(), schulung["fehler_2026"].to_numpy(), schulung["teile_2026"].to_numpy()


def gewichtetes_verhaeltnis(w, d=schulung):
    rate = lambda g: (w * g * d["fehler_2026"]).sum() / (w * g * d["teile_2026"]).sum()
    return rate(d["schulung"]) / rate(1 - d["schulung"])


print(list(merkmale.columns))
print(round(gewichtetes_verhaeltnis(schulung["w_ate"]), 3), round(schulung["w_ate"].max(), 2))

### TODO 6 · Schichten des Propensity Score
Statt zu gewichten kannst du nach dem Score stratifizieren: Personen mit ähnlichem Score vergleichen und die Unterschiede mitteln, wie in Abschnitt 3.1 des Skripts. Die Gruppen heißen hier Score-Schichten; mit der Arbeitsschicht haben sie nichts zu tun.
1. `ps_schicht`: die Score-Schicht jeder Person als Zahl von 0 bis 4 aus `pd.qcut(schulung["ps"], 5, labels=False)`.
2. `anzahl_schicht`: eine Tabelle mit einer Zeile je Score-Schicht und den Spalten `0` (ungeschult) und `1` (geschult), etwa mit `pd.crosstab`.
3. `verh_schicht`: je Score-Schicht das Verhältnis der Fehlerraten geschult zu ungeschult, jede Rate als Summe der Fehler 2026 durch Summe der Teile 2026, als Series mit den Score-Schichten als Index.
4. `verh_strat_att` und `verh_strat_ate`: das Mittel der Logarithmen von `verh_schicht`, gewichtet mit der Zahl der Geschulten beziehungsweise aller Personen je Score-Schicht, mit `np.exp` zurückgerechnet.

In [ ]:
# TODO 6
ps_schicht = None
anzahl_schicht = None
verh_schicht = None
verh_strat_att = None
verh_strat_ate = None

print(None if anzahl_schicht is None else anzahl_schicht.T)
print(None if verh_schicht is None else verh_schicht.round(3).tolist(), None if verh_strat_att is None else round(verh_strat_att, 3),
      None if verh_strat_ate is None else round(verh_strat_ate, 3))

In [ ]:
check("TODO 6 Score-Schichten", lambda: anzahl_schicht[1].tolist() == [3, 1, 5, 11, 14] and anzahl_schicht[0].tolist() == [22, 24, 19, 14, 11])
check("TODO 6 Verhältnisse", lambda: np.allclose(verh_schicht, [1.077, 1.1913, 0.7121, 0.6797, 0.9628], atol=5e-4)
      and abs(verh_strat_att - 0.8363) < 5e-4 and abs(verh_strat_ate - 0.904) < 5e-4)

In den beiden unteren Score-Schichten sitzen nur 3 und 1 Geschulte, in den beiden oberen 11 und 14. Die Verhältnisse je Score-Schicht schwanken von 0,680 bis 1,191; in den unteren Schichten stützt sich jedes auf sehr wenige Geschulte. Gewichtet mit den Geschulten ergibt sich 0,836, nah an der Regression. Gewichtet mit allen Personen ergibt sich 0,904, weil die unteren Schichten mit kaum Geschulten und Verhältnissen über 1 dann viel Gewicht bekommen. Die beiden Mittel beantworten verschiedene Fragen: das erste die Wirkung bei den Geschulten, das zweite die Wirkung bei allen.

### TODO 7 · Gewichte für die Geschulten und Balance
Für die Wirkung bei den Geschulten (ATT) bekommen Geschulte das Gewicht 1 und Ungeschulte $e/(1-e)$. Dann sehen die gewichteten Ungeschulten aus wie die Geschulten.
1. `schulung["w_att"]`: die ATT-Gewichte als neue Spalte, mit `schulung["ps"]` als $e$.
2. `balance_att`: eine Series mit den Spalten von `merkmale` als Index und der standardisierten Differenz nach dieser Gewichtung, `smd(merkmale[c], T, schulung["w_att"])`.

In [ ]:
# TODO 7
# schulung["w_att"] = ...
balance_att = None

print(round(schulung["w_att"].max(), 2) if "w_att" in schulung else None, None if balance_att is None else balance_att.round(3).to_dict())

In [ ]:
check("TODO 7 Gewichte", lambda: "w_att" in schulung and bool((schulung.loc[T == 1, "w_att"] == 1).all())
      and abs(schulung["w_att"].max() - 2.347) < 5e-4 and abs(schulung.loc[T == 0, "w_att"].sum() - 34.2793) < 5e-3)
check("TODO 7 Balance", lambda: list(balance_att.index) == list(merkmale.columns) and np.allclose(balance_att, [0.0447, -0.0669, -0.0236, -0.012, 0.0578, 0.096, -0.0614, 0.0154], atol=5e-4)
      and bool((balance_att.abs() < 0.1).all()))

In [ ]:
# Vorgabe: Ist TODO 7 noch offen, legt diese Zelle schulung["w_att"] an. In der Lösung ändert sie nichts.
if "w_att" not in schulung:
    schulung["w_att"] = T + (1 - T) * schulung["ps"] / (1 - schulung["ps"])
print(round(schulung["w_att"].sum(), 1))

Das größte ATT-Gewicht ist 2,35, viel kleiner als das größte ATE-Gewicht von 15,72 im Skript: Geschulte zählen einfach, und der größte Score unter den Ungeschulten ist 0,701. Die Gewichte der 90 Ungeschulten summieren sich zu 34,3, fast genau zur Zahl der Geschulten, wie es bei ATT-Gewichten im Mittel sein soll. Alle acht standardisierten Differenzen liegen unter 0,1. Am größten ist der Springer-Anteil mit 0,096, die Nachtschicht liegt bei 0,045 statt −0,188 nach der ATE-Gewichtung im Skript, Erfahrung und Fehlerquote 2025 bei −0,061 und 0,015. Für die Geschulten gelingt der Ausgleich hier besser als für alle.

### TODO 8 · IPW für die Geschulten mit Bootstrap
1. `ipw_att`: das gewichtete Verhältnis der Fehlerraten mit `gewichtetes_verhaeltnis(schulung["w_att"])` als Zahl.
2. `ki_att`: das 95-%-Intervall als Array aus zwei Zahlen, aus einem Bootstrap über Personen genau wie in Abschnitt 4.5 des Skripts, nur mit den ATT-Gewichten: `rng = np.random.default_rng(SEED)`, dann `ziehung = rng.integers(0, len(schulung), size=(2000, len(schulung)))`, die Matrix `anzahl` mit `np.bincount` je Zeile, die gewichteten Raten je Wiederholung und `np.quantile(..., [0.025, 0.975])`. Der Propensity Score bleibt fest.

In [ ]:
# TODO 8
rng = np.random.default_rng(SEED)
ipw_att = None
ki_att = None

print(None if ipw_att is None else round(ipw_att, 3), None if ki_att is None else np.round(ki_att, 3))

In [ ]:
check("TODO 8 IPW für die Geschulten", lambda: abs(ipw_att - 0.8736) < 5e-4)
check("TODO 8 Bootstrap-Intervall", lambda: len(ki_att) == 2 and ki_att[0] < ipw_att < ki_att[1] and ki_att[0] < 1 < ki_att[1]
      and abs(ki_att[0] - 0.678) < 0.025 and abs(ki_att[1] - 1.122) < 0.035)

In [ ]:
# Vorgabe: Ist TODO 8 noch offen, setzt diese Zelle die gerundeten Werte der Lösung ein. In der Lösung ändert sie nichts.
if ipw_att is None or ki_att is None:
    ipw_att, ki_att = 0.874, np.array([0.664, 1.137])
print(round(ipw_att, 3), np.round(ki_att, 3))

Mit den ATT-Gewichten ist das Verhältnis 0,874, das 95-%-Intervall reicht von 0,664 bis 1,137 und schließt 1 ein. Mit einem anderen Seed verschieben sich die Grenzen um etwa ein bis drei Hundertstel. Wie im Skript ist das Intervall viel breiter als das der Regression, 0,788 bis 0,920: Die Gewichtung nutzt die Fehlerquote 2025 nur zum Angleichen der Gruppen, die Regression auch, um die Streuung der Fehler 2026 zu erklären. Trotz der kleineren Gewichte ist es sogar etwas breiter als das Intervall für den ATE im Skript, 0,687 bis 1,098. Kleine Gewichte allein machen eine Schätzung nicht genau.

### TODO 9 · Matching mit drei Partnern
Im Skript bekommt jede geschulte Person einen Partner, und die Erfahrung ist danach schlechter ausgeglichen als vorher. Mit drei Partnern je Person wird der Vergleich ruhiger.
1. Für jede geschulte Person die drei Ungeschulten mit dem kleinsten Abstand im Propensity Score, mit Zurücklegen. Nimm die Abstandsmatrix wie im Skript und `np.argsort(abstand, axis=1, kind="stable")[:, :3]`.
2. `gew_partner`: ein Array mit einem Gewicht je Person: wie oft sie als Partner gewählt wurde, geteilt durch 3. Geschulte haben das Gewicht 0.
3. `match3`: das Verhältnis der Fehlerraten der Geschulten zu ihren gewichteten Partnern, jede Rate als Summe der Fehler durch Summe der Teile.
4. `balance_match3`: die standardisierten Differenzen aller Spalten von `merkmale` mit `smd(merkmale[c], T, T + gew_partner)`.

In [ ]:
# TODO 9
gew_partner = None
match3 = None
balance_match3 = None

print(None if gew_partner is None else int((np.asarray(gew_partner) > 0).sum()), None if match3 is None else round(match3, 3))
print(None if balance_match3 is None else balance_match3.round(3).to_dict())

In [ ]:
check("TODO 9 Partner und Verhältnis", lambda: len(gew_partner) == len(schulung) and abs(np.sum(gew_partner) - 34) < 1e-9
      and float(np.sum(np.asarray(gew_partner)[T == 1])) == 0 and int((np.asarray(gew_partner) > 0).sum()) == 43 and abs(match3 - 0.8805) < 5e-4)
check("TODO 9 Balance", lambda: list(balance_match3.index) == list(merkmale.columns) and np.allclose(balance_match3, [-0.1579, 0.0202, 0.1169, 0.0802, 0.0449, 0.1276, -0.0862, -0.0112], atol=5e-4))

Mit drei Partnern je Person kommen 43 verschiedene Ungeschulte zum Zug, mit einem Partner waren es 23. Das Verhältnis ist 0,880, nah an Gewichtung und Regression; mit einem Partner war es 0,963. Die Erfahrung ist mit −0,086 jetzt ausgeglichen, mit einem Partner lag sie bei −0,315 (für die abgeschnittene Erfahrung). Die Fehlerquote 2025 liegt bei −0,011. Über der Grenze von 0,1 bleiben die Nachtschicht mit −0,158, die Springer mit 0,128 und Linie 2 mit 0,117. Mehr Partner stützen den Vergleich auf mehr Personen, dafür sind die Partner im Mittel etwas weniger ähnlich. Prüfe nach jedem Matching die Balance.

## 5 · Differenz-in-Differenzen
*Lerneinheit 5 · Skript Abschnitt 5*

Die **Differenz der Differenzen** vergleicht die Änderung an Linie 2 mit der Änderung an den Vergleichslinien 1 und 3. Unter der Annahme **paralleler Trends** misst die Änderung der Vergleichslinien, was an Linie 2 ohne Workshop passiert wäre. Die nächste Zelle lädt die Rüstvorgänge wie im Skript: `vor` und `nach` markieren Mai bis Oktober 2025 und Dezember 2025 bis Mai 2026, `zwei` enthält die Linien 1 bis 3 in beiden Zeiträumen mit `nachher` und `linie_2`. Danach kommen Kalenderwoche, große Wechsel, der Workshop-Tag `WORKSHOP`, die Tabelle `d13` mit den Linien 1 bis 3 und die Funktion `did` für die Regression mit Linien- und Wochen-Effekten. `did` gibt Schätzung und Standardfehler auf der Log-Skala zurück.

In [ ]:
ruesten = pd.read_csv("daten/ruestvorgaenge.csv", parse_dates=["datum"])
ruesten["log_dauer"] = np.log(ruesten["dauer_min"])
vor = ruesten["datum"].between("2025-05-01", "2025-10-31")
nach = ruesten["datum"].between("2025-12-01", "2026-05-31")
zwei = ruesten[(vor | nach) & ruesten["linie"].isin([1, 2, 3])].assign(nachher=nach, linie_2=lambda d: d["linie"] == 2)
ruesten["woche"] = ruesten["datum"].dt.to_period("W-SUN").astype(str)
ruesten["gross"] = (ruesten["wechsel"] == "groß").astype(int)
WORKSHOP = pd.Timestamp("2025-11-03")
d13 = ruesten[ruesten["linie"].isin([1, 2, 3])].copy()
d13["workshop"] = ((d13["linie"] == 2) & (d13["datum"] >= WORKSHOP)).astype(int)


def did(daten):
    m = smf.ols("log_dauer ~ workshop + C(linie) + C(woche) + gross", data=daten).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(daten["woche"])[0]})
    return round(float(m.params["workshop"]), 4), round(float(m.bse["workshop"]), 4)


print(len(ruesten), len(zwei), did(d13))

In [ ]:
verlauf = (ruesten.assign(monat=ruesten["datum"].dt.to_period("M").dt.to_timestamp())
           .pivot_table(index="monat", columns="linie", values="log_dauer", aggfunc="mean"))
fig, ax = plt.subplots(figsize=(9, 4))
for linie in verlauf.columns:
    ax.plot(verlauf.index, verlauf[linie], marker="o", markersize=3, label=f"Linie {linie}")
ax.axvline(WORKSHOP, color="grey", linestyle="--")
ax.axvline(pd.Timestamp("2026-01-12"), color="grey", linestyle=":")
ax.set_ylabel("Mittel der logarithmierten Rüstdauer")
ax.legend(ncol=4)
plt.show()

Die gestrichelte Linie markiert den Workshop, die gepunktete den Start des neuen Auftragssystems am 12. Januar 2026.

### TODO 10 · Differenz-in-Differenzen nach Art des Wechsels
Große Wechsel dauern länger als kleine. Verkürzt der Workshop beide Arten gleich stark?
1. `dd_wechsel`: eine Tabelle mit den Zeilen `groß` und `klein` und den Spalten `dd` (Differenz der Differenzen der logarithmierten Dauer aus den vier Mittelwerten wie im Skript, nur mit den Rüstvorgängen dieser Art aus `zwei`), `faktor` ($e^{dd}$) und `n` (Zahl der Rüstvorgänge dieser Art in `zwei`).

In [ ]:
# TODO 10
dd_wechsel = None

print(None if dd_wechsel is None else dd_wechsel.round(3))

In [ ]:
check("TODO 10 Tabelle", lambda: list(dd_wechsel.index) == ["groß", "klein"] and list(dd_wechsel["n"]) == [680, 1491])
check("TODO 10 Differenz der Differenzen", lambda: np.allclose(dd_wechsel["dd"].astype(float), [-0.2803, -0.2779], atol=5e-4)
      and np.allclose(dd_wechsel["faktor"].astype(float), np.exp(dd_wechsel["dd"].astype(float))))

Bei den 680 großen Wechseln ist die Differenz der Differenzen −0,280, als Faktor 0,756. Bei den 1.491 kleinen Wechseln ist sie −0,278, als Faktor 0,757. Unter der Annahme paralleler Trends verkürzt der Workshop beide Arten etwa gleich stark, um ein knappes Viertel. Getrennt liegen beide etwas weiter von null als die −0,255 über alle Wechsel im Skript: Nach dem Workshop kommen an Linie 2 etwas mehr große Wechsel vor, und die dauern länger. Diese Verschiebung steckt im Vergleich über alle Wechsel mit drin. Die Regression im Skript nimmt deshalb die Art des Wechsels als Kovariate auf.

### TODO 11 · Vorher-nachher an Linie 2 und das Auftragssystem
Vom 12. Januar 2026 an lief das neue System für Fertigungsaufträge, und in den Wochen danach dauert Rüsten überall länger. Wie viel der Verzerrung von vorher-nachher geht auf diese Phase zurück?
1. `systemphase`: eine boolesche Series über `ruesten`, wahr für die acht Wochen vom 12. Januar bis 8. März 2026. Vergleiche mit der Datumsspalte, nicht mit Text.
2. `system_faktor`: der Faktor auf die Dauer in der Systemphase an den Linien 1, 3 und 4 von Dezember 2025 bis Mai 2026 (`nach`). Nimm eine Spalte `system` aus 0 und 1, das Modell `smf.ols("log_dauer ~ system + C(linie) + gross", ...)` und rechne den Koeffizienten mit `np.exp` zurück.
3. `vn_ohne_system`: vorher-nachher an Linie 2 auf der Log-Skala (Mittel nachher minus Mittel vorher, Zeiträume wie in `zwei`) ohne die Systemphase. `dd_ohne_system`: die Differenz der Differenzen mit den Linien 1 und 3, ebenfalls ohne die Systemphase.

In [ ]:
# TODO 11
systemphase = None
system_faktor = None
vn_ohne_system = None
dd_ohne_system = None

print(None if systemphase is None else int(systemphase.sum()), None if system_faktor is None else round(system_faktor, 3))
print(None if vn_ohne_system is None else round(vn_ohne_system, 3), None if dd_ohne_system is None else round(dd_ohne_system, 3))

In [ ]:
check("TODO 11 Systemphase", lambda: int(systemphase.sum()) == 494 and abs(system_faktor - 1.1103) < 5e-4)
check("TODO 11 ohne Systemphase", lambda: abs(vn_ohne_system - (-0.1993)) < 5e-4 and abs(dd_ohne_system - (-0.2712)) < 5e-4)

In den acht Wochen der Systemphase dauert Rüsten an den Linien ohne Workshop etwa 11 % länger, Faktor 1,110. Ohne diese Wochen sinkt die Dauer an Linie 2 auf der Log-Skala um −0,199 statt um −0,156. Die Vergleichslinien bleiben aber auch ohne Systemphase um 0,072 langsamer als vor dem Workshop. Die Differenz der Differenzen ohne Systemphase ist −0,271, nah an den −0,255 mit allen Wochen. Die Systemphase erklärt also nur einen Teil der Verzerrung von vorher-nachher. Wer sie nur bei Linie 2 herausnimmt und vorher-nachher als Wirkung berichtet, übersieht den Rest, den die Vergleichslinien sichtbar machen.

## 6 · Annahmen prüfen: Trends, Placebos, Sensitivität
*Lerneinheit 6 · Skript Abschnitt 6*

Ein **Placebo-Test** wendet dieselbe Auswertung dort an, wo es keine Wirkung geben kann. Im Skript liefert ein erfundener Workshop-Tag im April 2025 eine unauffällige Schätzung von 0,0284. Jetzt prüfst du Linien ohne Workshop. Die Regression mit Wochen-Effekten dauert im Browser je Aufruf etwa eine halbe Sekunde.

### TODO 12 · Placebo-Linien
Die Funktion `did` braucht eine Tabelle mit den Spalten `log_dauer`, `workshop`, `linie`, `woche` und `gross`.
1. `placebo_linien`: ein Dictionary mit den Schlüsseln 1, 3 und 4. Für jede dieser Linien nimmst du alle Rüstvorgänge der Linien 1, 3 und 4 (ohne Linie 2), setzt `workshop` auf 1 für diese Linie ab `WORKSHOP` und sonst auf 0 und speicherst das Ergebnis von `did`.
2. `placebo_3_1`: dasselbe nur mit den Linien 1 und 3, Linie 3 als scheinbar behandelt.

In [ ]:
# TODO 12
placebo_linien = None
placebo_3_1 = None

print(placebo_linien, placebo_3_1)

In [ ]:
check("TODO 12 Placebo-Linien", lambda: sorted(placebo_linien) == [1, 3, 4]
      and np.allclose([placebo_linien[l] for l in (1, 3, 4)], [[-0.0736, 0.0247], [-0.0926, 0.0217], [0.1626, 0.0247]], atol=5e-4))
check("TODO 12 Linie 3 gegen Linie 1", lambda: np.allclose(placebo_3_1, [-0.0162, 0.026], atol=5e-4))

Als scheinbar behandelt liefert Linie 1 −0,074 mit Standardfehler 0,025, Linie 3 −0,093 mit 0,022 und Linie 4 +0,163 mit 0,025. Linie 3 gegen Linie 1 allein ergibt −0,016 mit 0,026, unauffällig. Ohne Linie 4 im Vergleich bleibt also fast nichts übrig: Die Werte für Linie 1 und 3 spiegeln vor allem, dass Linie 4 nach dem Workshop langsamer wird. Linie 4 fällt aus der Reihe und verzerrt jeden Vergleich, in dem sie steckt. Gemessen an den geklumpten Standardfehlern wären auch −0,074 und −0,093 deutlich von null verschieden. Ganze Linien weichen also stärker voneinander ab, als Standardfehler nach Woche erwarten lassen. Die echte Schätzung von −0,237 liegt trotzdem weit außerhalb.

Die nächste Zelle legt die Funktion `e_wert` aus Abschnitt 6.4 des Skripts an. Für ein Verhältnis unter 1 dreht sie es zuerst um.

In [ ]:
def e_wert(rr):
    """E-Wert nach VanderWeele und Ding (2017) für ein Verhältnis; unter 1 wird es zuerst umgedreht."""
    rr = 1 / rr if rr < 1 else rr
    return rr + np.sqrt(rr * (rr - 1))


print(round(e_wert(0.851), 3), round(e_wert(0.920), 3))

## 7 · Ehrlich berichten
*Lerneinheit 7 · Skript Abschnitt 7*

Ein Ergebnisbericht nennt die geschätzte Größe, das Verfahren, das Intervall und die Annahmen, unter denen die Schätzung kausal gilt. Im Skript steht er als Tabelle und Text. Hier legst du ihn als Objekt an, das sich prüfen und weiterverarbeiten lässt.

### TODO 13 · E-Wert und Ergebnisbericht
1. `e_att`: ein Dictionary mit den Schlüsseln `"Schätzung"` und `"Grenze"`: der E-Wert mit `e_wert` für `ipw_att` und für die Grenze von `ki_att`, die näher an 1 liegt. Schließt das Intervall die 1 ein, ist der E-Wert der Grenze 1.
2. `bericht`: ein Dictionary mit den Schlüsseln `"Schulung"` und `"Workshop"`. Jeder Eintrag ist wieder ein Dictionary mit `"Schätzung"`, `"unten"`, `"oben"`, `"Verfahren"` und `"Annahme"`. Für die Schulung nimmst du `verhaeltnis` mit allen Störgrößen, für den Workshop `b, se = did(d13)` als Faktor $e^{b}$ mit Intervall $e^{b \pm 1{,}96 \cdot se}$. Verfahren und Annahme schreibst du als kurzen Text.

In [ ]:
# TODO 13
e_att = None
bericht = None

print(None if e_att is None else {k: round(float(v), 3) for k, v in e_att.items()})
print(None if bericht is None else {k: [round(float(v[s]), 3) for s in ["Schätzung", "unten", "oben"]] for k, v in bericht.items()})

In [ ]:
check("TODO 13 E-Wert", lambda: set(e_att) == {"Schätzung", "Grenze"} and abs(e_att["Schätzung"] - e_wert(ipw_att)) < 1e-9
      and abs(e_att["Schätzung"] - 1.5517) < 0.005 and e_att["Grenze"] == 1)
check("TODO 13 Bericht", lambda: set(bericht) == {"Schulung", "Workshop"}
      and all(set(v) == {"Schätzung", "unten", "oben", "Verfahren", "Annahme"} and len(v["Verfahren"]) > 10 and len(v["Annahme"]) > 10 for v in bericht.values())
      and np.allclose([bericht["Schulung"][s] for s in ["Schätzung", "unten", "oben"]], [0.851, 0.788, 0.92], atol=6e-4)
      and np.allclose([bericht["Workshop"][s] for s in ["Schätzung", "unten", "oben"]], [0.7888, 0.7556, 0.8233], atol=6e-4))

Für die IPW-Schätzung 0,874 ist der E-Wert 1,552. Die obere Grenze des Intervalls liegt über 1, ihr E-Wert ist deshalb 1: Schon ohne unbeobachtete Störgröße ist eine Wirkung von null mit der Gewichtung vereinbar. Im Bericht steht die Schulung mit 0,851 (0,788 bis 0,920) aus der Poisson-Regression unter der Annahme, dass keine weitere Störgröße die Auswahl mit den Fehlern verbindet. Der Workshop steht mit 0,789 (0,756 bis 0,823) aus der Differenz-in-Differenzen unter der Annahme paralleler Trends. Die Placebo-Linien aus TODO 12 erinnern daran, dass das Intervall des Workshops eher zu schmal ist. Auch das gehört als Satz zur Unsicherheit in den Bericht.

### Bonus 1 · Linie 4 als Vergleich
1. `dd_linie4`: die Differenz der Differenzen aus den vier Mittelwerten mit Linie 4 allein als Vergleich, Zeiträume `vor` und `nach` wie in `zwei`.
2. `steigung_vorher`: ein Dictionary mit den Schlüsseln 2 und 4. Darin steht die Steigung (`np.polyfit(..., 1)[0]`) des monatlichen Abstands dieser Linie zum Mittel der Linien 1 und 3, über die 13 Monate vor dem Workshop (Oktober 2024 bis Oktober 2025) mit den Monaten als 0, 1, 2 und so weiter. Den Abstand je Monat rechnest du aus den Monatsmitteln der logarithmierten Dauer wie in Abschnitt 6.2 des Skripts.

In [ ]:
# Bonus 1
dd_linie4 = None
steigung_vorher = None

print(None if dd_linie4 is None else round(dd_linie4, 3), None if steigung_vorher is None else {k: round(float(v), 4) for k, v in steigung_vorher.items()})

In [ ]:
check("Bonus 1 Linie 4 als Vergleich", lambda: abs(dd_linie4 - (-0.327)) < 5e-4 and set(steigung_vorher) == {2, 4}
      and abs(steigung_vorher[2] - 0.00949) < 5e-5 and abs(steigung_vorher[4] - 0.01816) < 5e-5)

Mit Linie 4 allein als Vergleich ist die Differenz der Differenzen −0,327, als Faktor 0,721, statt −0,255 mit den Linien 1 und 3. Schon vor dem Workshop wächst der Abstand von Linie 4 zu den Linien 1 und 3 um etwa 0,018 je Monat, der von Linie 2 um 0,0095, halb so schnell. Wird die Vergleichsgruppe aus eigenen Gründen langsamer, schreibt die Differenz der Differenzen diese Verlangsamung dem Workshop gut. Prüfe die Trends vor der Maßnahme für jede Vergleichsgruppe einzeln.

### Bonus 2 · Gewichte kürzen
Große Gewichte machen IPW unsicher. Man kann sie bei einer Grenze kürzen und tauscht etwas Verzerrung gegen weniger Streuung.
1. `gekuerzt`: ein Dictionary mit den Schlüsseln 10 und 5. Zu jeder Grenze gehört ein Tupel aus der Zahl der ATE-Gewichte über der Grenze und dem gewichteten Verhältnis mit `gewichtetes_verhaeltnis(schulung["w_ate"].clip(upper=grenze))`.

In [ ]:
# Bonus 2
gekuerzt = None

print(None if gekuerzt is None else {k: (v[0], round(float(v[1]), 3)) for k, v in gekuerzt.items()})

In [ ]:
check("Bonus 2 gekürzte Gewichte", lambda: set(gekuerzt) == {10, 5} and gekuerzt[10][0] == 3 and gekuerzt[5][0] == 5
      and abs(gekuerzt[10][1] - 0.8811) < 5e-4 and abs(gekuerzt[5][1] - 0.9022) < 5e-4)

3 Gewichte liegen über 10, gekürzt ist das Verhältnis 0,881. 5 Gewichte liegen über 5, gekürzt ist es 0,902. Ungekürzt war es 0,866. Kürzen zieht die Schätzung in Richtung des naiven Vergleichs, denn die gekürzten Gewichte gehören zu Geschulten mit kleinem Score, die ohne Kürzung für viele ähnliche Personen stehen. Welche Grenze man nimmt, legt man vor der Auswertung fest und berichtet die ungekürzte Schätzung mit.

## Zum Nachdenken

1. In TODO 2 sind die Geschulten in beiden Hälften besser, insgesamt schlechter. Woher weißt du, welcher Vergleich gilt?
2. In TODO 3 bleibt ohne Fehler 2025 genau eine Hintertür offen, über das unbeobachtete Können. Was wäre, wenn das Können auch direkt auf die Auswahl gewirkt hätte, etwa weil die Schichtleitungen ihre Leute kennen?
3. Für die Geschulten liefern Score-Schichten, Gewichtung und Matching Werte zwischen 0,836 und 0,880, die Regression 0,851, mit sehr verschiedenen Intervallen. Welche Zahl schreibst du in den Bericht?
4. Die Placebo-Linien in TODO 12 liegen deutlich weiter von null, als ihre Standardfehler erwarten lassen. Was folgt daraus für das Intervall des Workshops?
5. Die Werksleitung möchte die Schulung allen anbieten, die 2025 viele Fehler hatten. Was sagt der ATT dazu, was der ATE?

<details><summary>Hinweise zu den Antworten</summary>

1. Aus den Annahmen, nicht aus den Daten. Die Fehlerquote 2025 stand vor der Schulung fest und beeinflusst Auswahl und Fehler 2026, sie ist eine Störgröße. Deshalb gilt der Vergleich innerhalb der Hälften. Wäre sie eine Folge der Schulung, gälte der Vergleich insgesamt.
2. Dann gäbe es den Pfeil Können → Schulung, und der Pfad Schulung ← Können → Fehler 2026 bliebe offen, denn das Können ist nicht gemessen. Keine Adjustierung schließt ihn. Der E-Wert zeigt, wie stark eine solche Größe sein müsste, und ein Versuch mit verlosten Plätzen würde das Problem lösen.
3. Die Hauptschätzung, die vor der Auswertung festgelegt war, hier die Regression mit 0,851 und ihrem Intervall. Die übrigen Verfahren gehören als Prüfung dazu: Sie zeigen, dass die Richtung nicht am Verfahren hängt, und die breiten Intervalle von IPW zeigen, wie viel an der Annahme über die Form der Regression hängt.
4. Das Intervall mit Standardfehlern nach Woche ist zu schmal. Ehrlicher ist es, die Streuung der Placebo-Schätzungen als Maß für die Unsicherheit mitzuberichten. Die Wirkung bleibt deutlich, denn −0,237 liegt weit außerhalb aller Placebo-Werte.
5. Der ATT beschreibt die Wirkung bei denen, die tatsächlich geschult wurden, und die hatten 2025 eher viele Fehler. Er passt deshalb besser zu dieser Frage. Der ATE beschreibt die Wirkung, wenn alle geschult würden. Er hängt hier an wenigen großen Gewichten und ist unsicherer.
</details>

## Wie es weitergeht

Du hast die Gruppen vor der Schulung mit standardisierten Differenzen verglichen, das Simpson-Paradox in den Nordwerk-Daten gefunden und das Hintertürkriterium als Funktion geschrieben. Du hast in Prozentpunkten adjustiert, Mediator und Collider erkannt, nach dem Score stratifiziert, für die Geschulten gewichtet, einen Bootstrap gezogen und mit drei Partnern gematcht. Beim Workshop hast du nach Art des Wechsels getrennt, die Systemphase herausgerechnet, Placebo-Linien geprüft und einen Ergebnisbericht mit E-Wert angelegt. Das war das letzte Fachmodul. In **Modul 14 „Projektwerkstatt zu zweit“** wendest du die Werkzeuge des Kurses zu zweit an einer eigenen Frage an. Die Leitfrage dort: Wie wird aus zwei Analysen eine Entscheidungsvorlage, die die Abnahme besteht?